# MicroPhaseLab Lesson C: Interpreting a Classical Baseline

This notebook explains the Otsu-plus-morphology baseline on the real validation split. Before opening it, complete Lesson B and run:

```powershell
microphaselab baseline --manifest data/splits/val.csv --output-dir outputs/baseline/otsu_val
```

Launch JupyterLab from the project root with `jupyter lab`, then open this notebook. Run every cell in order. Use validation results to understand or select baseline settings; do not use test results for that purpose.

## Learning objectives

1. Read aggregate and per-image segmentation metrics.
2. Distinguish over-segmentation from under-segmentation.
3. Connect false positives and false negatives to the image locations that caused them.
4. Select a baseline setting using validation data only.

In [ ]:
from pathlib import Path

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

manifest_path = Path('../data/splits/val.csv')
baseline_root = Path('../outputs/baseline/otsu_val')

manifest = pd.read_csv(manifest_path)
metrics = pd.read_csv(baseline_root / 'metrics_per_image.csv')
summary = json.loads((baseline_root / 'summary.json').read_text(encoding='utf-8'))
rows = manifest.merge(metrics, on='image_id', validate='one_to_one')

print(f'Validation images: {len(rows)}')
{key: summary[key] for key in (
    'mean_dice', 'mean_iou', 'mean_precision', 'mean_recall',
    'mean_area_fraction_absolute_error',
)}

## Is the baseline predicting too much or too little MA?

Compare the expert MA fraction with the predicted MA fraction. If the predicted fraction is much larger, the method is over-segmenting and false positives are likely to dominate. If it is much smaller, the method is under-segmenting and false negatives are likely to dominate.

In [ ]:
mean_error_summary = metrics[[
    'target_fraction', 'prediction_fraction', 'false_positive', 'false_negative',
    'dice', 'iou', 'precision', 'recall',
]].mean().rename('validation_mean')
mean_error_summary.to_frame()

## Inspect per-image variation

The mean can hide difficult images. The table below lists the ten images with the lowest Dice score. Read precision and recall alongside Dice: low precision means extra predicted MA, while low recall means missed expert-labelled MA.

In [ ]:
display_columns = [
    'image_id', 'target_fraction', 'prediction_fraction', 'false_positive',
    'false_negative', 'dice', 'iou', 'precision', 'recall',
]
metrics.nsmallest(10, 'dice')[display_columns]

## Inspect the lowest-Dice prediction

The output mask files store 0 for background and 1 for predicted MA. A standard image viewer may display both values as almost black because it expects the range 0–255. The first three panels below use `vmin=0` and `vmax=1` so that MA is visibly white.

In the last panel, white is a correct MA prediction, red is a false positive (extra predicted MA), and blue is a false negative (missed MA).

In [ ]:
worst_row = rows.loc[rows['dice'].idxmin()]
image = np.asarray(Image.open(worst_row.image_path).convert('L'))
target = np.asarray(Image.open(worst_row.mask_path)) > 0
prediction = np.asarray(Image.open(worst_row.prediction_path)) > 0

true_positive = target & prediction
false_positive = ~target & prediction
false_negative = target & ~prediction
error_image = np.zeros((*target.shape, 3))
error_image[true_positive] = [1, 1, 1]
error_image[false_positive] = [1, 0, 0]
error_image[false_negative] = [0, 0.5, 1]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title(f'SEM image: {worst_row.image_id}')
axes[1].imshow(target, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Expert mask')
axes[2].imshow(prediction, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Baseline prediction')
axes[3].imshow(error_image)
axes[3].set_title('White: correct; red: FP; blue: FN')
for axis in axes:
    axis.axis('off')
plt.show()

worst_row[display_columns]

## Choose settings responsibly

If you compare baseline settings, change one parameter at a time and write each validation result to a new output directory. Select a setting using the metrics and visual errors together. Then freeze that setting before evaluating the test set once.

A high recall is not enough if the prediction labels most of the image as MA. A low area-fraction error is not enough if the predicted region is in the wrong location. The baseline is a transparent reference method, not a claim that global brightness thresholding is sufficient for real microstructures.

## Questions and suggested answers

1. **How do you identify over-segmentation?** The predicted MA fraction is much larger than the expert MA fraction, precision is low, and red false-positive regions dominate the error image.
2. **How do you identify under-segmentation?** The predicted MA fraction is much smaller than the expert fraction, recall is low, and blue false-negative regions dominate the error image.
3. **Why is the test split not used to choose a parameter?** Trying settings on the test split lets its results influence the choice, so it no longer measures performance on unseen data.
4. **What does a weak baseline teach us?** It establishes an honest reference. A U-Net must be compared against it on the same split, while still being judged for its own false-positive and false-negative errors.